In [33]:
import os
print(os.getcwd())

c:\Users\Manmeet\churn prediction\notebooks


In [34]:
import pandas as pd

In [35]:
df=pd.read_csv("C:\\Users\\Manmeet\\churn prediction\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [36]:
print(df.shape)

(7043, 21)


In [37]:
print(df["Churn"].value_counts(normalize=True))

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [38]:
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [39]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [40]:
df=df.dropna()

In [41]:
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})

In [42]:
X = df.drop(["Churn", "customerID"], axis=1)
y=df["Churn"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42
    ))
])

In [45]:
model.fit(X_train, y_train)

print("Train Accuracy:", model.score(X_train, y_train))
print("Test Accuracy:", model.score(X_test, y_test))

from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Train Accuracy: 0.8378666666666666
Test Accuracy: 0.7931769722814499
ROC AUC: 0.8347099202261209


In [46]:
import numpy as np

importances = model.named_steps["classifier"].feature_importances_
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

feature_data = sorted(
    zip(feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

for f, i in feature_data[:10]:
    print(f, i)

num__tenure 0.14945683792320297
cat__Contract_Month-to-month 0.12696761147902424
num__TotalCharges 0.1094095871609215
num__MonthlyCharges 0.06843658844460702
cat__OnlineSecurity_No 0.06336350440726493
cat__InternetService_Fiber optic 0.05752470102724963
cat__TechSupport_No 0.049352043184830344
cat__PaymentMethod_Electronic check 0.044856658377085794
cat__Contract_Two year 0.04250325977547794
cat__OnlineBackup_No 0.0204173030291255


In [47]:
import joblib

joblib.dump(model, "../model/churn_model.pkl")

['../model/churn_model.pkl']

In [48]:
print(list(X.columns))

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [49]:
import sklearn
print("Sklearn version:", sklearn.__version__)

Sklearn version: 1.6.1


In [50]:
model = joblib.load("../model/churn_model.pkl")

In [51]:
import sys
print(sys.executable)

C:\Users\Manmeet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
